# 🤖 AI Agent Swarm V2 — Kaggle Setup

**GPUs:** 2× T4 (32GB VRAM total)  
**LLMs:** Todos via [g4f.dev](https://g4f.dev) — **uma única API key**  
**Geração:** Z-Image-Turbo (imagem) + LTX-2.3 (vídeo) — local na GPU  

---
## ✅ Checklist antes de rodar
- [ ] GPU ativada: **Settings ⚙️ > Accelerator > GPU T4 × 2**
- [ ] Secrets configurados: `G4F_API_KEY`, `DISCORD_TOKEN`, `DISCORD_GUILD_ID`, `HF_TOKEN`
- [ ] Repositório no GitHub com o código do swarm
- [ ] Bot adicionado ao servidor Discord
---

In [ ]:
# ── Célula 1: Verificar GPU ───────────────────────────────────────────────────
import torch

print(f'CUDA disponível: {torch.cuda.is_available()}')
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} — {p.total_memory/1e9:.1f}GB')

if not torch.cuda.is_available():
    print('⚠️  GPU não detectada! Vá em Settings > Accelerator > GPU T4 × 2')

In [ ]:
# ── Célula 2: Instalar dependências ──────────────────────────────────────────
import subprocess, sys

packages = [
    'discord.py>=2.3.0',
    'diffusers>=0.33.0',
    'transformers>=4.52.0',
    'accelerate>=1.6.0',
    'imageio[ffmpeg]',
    'Pillow>=11.0.0',
    'safetensors',
    'sentencepiece',
    'huggingface_hub>=0.31.0',
    'python-dotenv',
]

for pkg in packages:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print(f'{'✅' if r.returncode == 0 else '❌'} {pkg}')

# ffmpeg — necessário para analyse_video
r = subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'],
                   capture_output=True, text=True)
print(f'{'✅' if r.returncode == 0 else '❌'} ffmpeg')
print('\nDependências instaladas.')

In [ ]:
# ── Célula 3: Carregar Secrets do Kaggle ─────────────────────────────────────
# Para adicionar: clique no ⚙️ > Add-ons > Secrets > Add a new secret
import os

try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()

    os.environ['G4F_API_KEY']      = s.get_secret('G4F_API_KEY')
    os.environ['DISCORD_TOKEN']    = s.get_secret('DISCORD_TOKEN')
    os.environ['DISCORD_GUILD_ID'] = s.get_secret('DISCORD_GUILD_ID')

    try:
        os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
        print('✅ HF_TOKEN carregado')
    except Exception:
        print('⚠️  HF_TOKEN ausente — geração de imagem/vídeo desativada')

    print('✅ Secrets carregados com sucesso')
    print(f'   G4F_API_KEY:  {os.environ["G4F_API_KEY"][:14]}...')
    print(f'   DISCORD:      {os.environ["DISCORD_TOKEN"][:12]}...')

except Exception as e:
    print(f'⚠️  Secrets automáticos falharam ({e})')
    print('Defina manualmente abaixo se necessário:')
    # os.environ['G4F_API_KEY']      = 'g4f_...'
    # os.environ['DISCORD_TOKEN']    = '...'
    # os.environ['DISCORD_GUILD_ID'] = '...'

In [ ]:
# ── Célula 4: Clonar repositório do GitHub ───────────────────────────────────
import subprocess, os

GITHUB_REPO = 'https://github.com/SEU_USUARIO/ai-swarm-v2.git'  # ← edite
DEST        = '/kaggle/working/swarm'

if os.path.exists(DEST):
    subprocess.run(['git', '-C', DEST, 'pull'], capture_output=True)
    print('🔄 Repositório atualizado')
else:
    r = subprocess.run(['git', 'clone', GITHUB_REPO, DEST],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'❌ Erro: {r.stderr[:200]}')
    else:
        print(f'✅ Clonado em {DEST}')

os.chdir(DEST)
print(f'📁 Diretório: {os.getcwd()}')

In [ ]:
# ── Célula 5: Probe dos modelos via g4f.dev ───────────────────────────────────
import sys
sys.path.insert(0, '/kaggle/working/swarm')

from llm_engine import probe_model
from config import G4F_MODELS

print('🔬 Testando modelos via g4f.dev...\n')
print(f'{"Modelo":<20} {"Status":<8} {"ms":<8} Preview')
print('─' * 65)

working = []
for key in G4F_MODELS:
    r    = probe_model(key)
    icon = '✅' if r['ok'] else '❌'
    print(f"{key:<20} {icon} {'OK' if r['ok'] else 'FAIL':<6} {r['latency_ms']:>5}ms  {r['response_preview'][:35]}")
    if r['ok']:
        working.append(key)

print(f'\n{len(working)}/{len(G4F_MODELS)} online: {working}')
if not working:
    print('⚠️  Nenhum modelo respondeu. Verifique G4F_API_KEY.')

In [ ]:
# ── Célula 6 (opcional): Pré-carregar modelos de geração na VRAM ──────────────
# Descomente para ter imagem/vídeo prontos imediatamente ao iniciar
# Tempo: ~3-5 min na primeira vez (baixa ~20GB do HuggingFace)

# import sys; sys.path.insert(0, '/kaggle/working/swarm')
# from media_engine import _load_zimage, _load_ltxv
# print('Carregando Z-Image-Turbo...')
# _load_zimage()
# print('Carregando LTX-2.3...')
# _load_ltxv()
# print('✅ Prontos na VRAM')

print('Pré-carga comentada. Descomente para ativar.')

In [ ]:
# ── Célula 7: Iniciar o bot Discord ──────────────────────────────────────────
# ⚠️  Esta célula fica rodando enquanto o bot estiver ativo.
#     Para parar: clique em ■ ao lado da célula.

import sys, os, threading, time
sys.path.insert(0, '/kaggle/working/swarm')
os.chdir('/kaggle/working/swarm')

# Mantém o notebook vivo (Kaggle desconecta após ~20 min sem output)
def _heartbeat():
    while True:
        time.sleep(60)
        print('💓', end='', flush=True)

threading.Thread(target=_heartbeat, daemon=True).start()

print('🚀 Iniciando AI Agent Swarm V2...')
print('   Teste no Discord: !help')
print('   Para parar: clique em ■ nesta célula\n')

exec(open('discord/bot.py').read())